In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup + NAR patch (standalone; no dependency on prior notebooks)
#
# NOTE: deliberately does NOT run `pip install casanovo`. That command
# re-resolves casanovo's depthcharge-ms pin and silently overwrites the
# editable depthcharge install, wiping out flash_compatible/new_zeros.
# ═══════════════════════════════════════════════════════════════════════
import os, sys, time, threading, warnings, inspect, datetime
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from contextlib import contextmanager
from tqdm import tqdm
from torch.profiler import profile, ProfilerActivity, schedule, record_function

torch.manual_seed(42)

# ── Directories ────────────────────────────────────────────────────────
WORK_DIR    = '/teamspace/studios/this_studio/nar_profiling'
RESULTS_DIR = '/teamspace/studios/this_studio/aoti_profiling/results'
AOTI_DIR    = '/teamspace/studios/this_studio/aoti_profiling/artifacts'
os.chdir(WORK_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(AOTI_DIR, exist_ok=True)

# ── Device / constants ─────────────────────────────────────────────────
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
BF16_DTYPE     = torch.bfloat16
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if DEVICE == 'cuda' else False

N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 10
N_PEAKS          = 150          # fixed peak budget — required for static AOTI shapes
PROF_WARMUP      = 20
PROF_ACTIVE      = 50

# ── depthcharge branch check ───────────────────────────────────────────
import depthcharge as _dc
from depthcharge.transformers import AnalyteTransformerDecoder, SpectrumTransformerEncoder

print(f'depthcharge: {_dc.__file__}')
# inspect.signature() reads the function's code object, NOT the source file —
# immune to the OSError that inspect.getsource() hits on PEP-660 editable installs.
_HAS_FLASH_COMPAT = 'flash_compatible' in inspect.signature(AnalyteTransformerDecoder.embed).parameters
print(f'  flash_compatible param available : {"✓" if _HAS_FLASH_COMPAT else "✗ (stock depthcharge)"}')
if not _HAS_FLASH_COMPAT:
    print('  → NAR patch will fall back to the all-False tgt_mask approach.')

# ── NAR patch (adaptive: uses flash_compatible if the branch provides it) ──
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep

_ar_embed_original = AnalyteTransformerDecoder.embed   # parent class → recursion-safe

if _HAS_FLASH_COMPAT:
    def _nar_embed(self, tokens, *args, memory,
                   memory_key_padding_mask=None, memory_mask=None,
                   tgt_mask=None, flash_compatible=False,
                   _orig=_ar_embed_original, **kwargs):
        # flash_compatible is named explicitly so the caller's value is absorbed
        # here rather than colliding with the flash_compatible=True below.
        return _orig(self, tokens, *args, memory=memory,
                     memory_key_padding_mask=memory_key_padding_mask,
                     memory_mask=memory_mask, flash_compatible=True, **kwargs)
else:
    def _nar_embed(self, tokens, *args, memory,
                   memory_key_padding_mask=None, memory_mask=None,
                   tgt_mask=None, _orig=_ar_embed_original, **kwargs):
        L = tokens.shape[1] + 1                     # +1 for prepended global token
        tgt_mask = torch.zeros((L, L), dtype=torch.bool, device=tokens.device)
        return _orig(self, tokens, *args, memory=memory,
                     memory_key_padding_mask=memory_key_padding_mask,
                     memory_mask=memory_mask, tgt_mask=tgt_mask, **kwargs)

assert _ar_embed_original is not _nar_embed, 'Patch captured itself — restart kernel.'
PeptideDecoder.embed = _nar_embed

def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    mzs, ints, precursors = mzs.to(dev), ints.to(dev), precursors.to(dev)
    memories, mem_masks = self.encoder(mzs, ints)
    zero_tokens = (torch.zeros_like(seqs.to(dev)) if seqs is not None
                   else torch.zeros((mzs.shape[0], self.max_peptide_len),
                                    dtype=torch.long, device=dev))
    scores = self.decoder(tokens=zero_tokens, memory=memories,
                          memory_key_padding_mask=mem_masks, precursors=precursors)
    return scores, seqs
Spec2Pep._forward_step = _nar_forward_step

def _nar_forward(self, batch): return self._forward_step(batch)
Spec2Pep.forward = _nar_forward

print('\n── NAR Patch Status ──────────────────────────────────────────')
print(f'  PeptideDecoder.embed  → _nar_embed        : {"✓" if PeptideDecoder.embed is _nar_embed else "✗"}')
print(f'  Spec2Pep._forward_step patched            : {"✓" if Spec2Pep._forward_step is _nar_forward_step else "✗"}')
print(f'  Spec2Pep.forward patched                  : {"✓" if Spec2Pep.forward is _nar_forward else "✗"}')
print(f'  mode                                      : {"flash_compatible=True" if _HAS_FLASH_COMPAT else "all-False tgt_mask"}')
print('──────────────────────────────────────────────────────────────')

# ── Shared helpers ─────────────────────────────────────────────────────
def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

@contextmanager
def _bf16_ctx():
    with torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
        yield

def _mark_step():
    """Required when chaining two separately torch.compile'd modules."""
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

def pad_or_select_to_fixed_peaks(mzs, ints, n=N_PEAKS):
    """Coerce variable-length spectra to exactly n peaks (AOTI needs static shapes)."""
    bs, L = mzs.shape
    if L == n:
        return mzs.contiguous(), ints.contiguous(), 0
    if L < n:
        return (torch.nn.functional.pad(mzs,  (0, n - L)).contiguous(),
                torch.nn.functional.pad(ints, (0, n - L)).contiguous(), 0)
    idx = ints.topk(n, dim=1).indices
    return mzs.gather(1, idx).contiguous(), ints.gather(1, idx).contiguous(), bs

def _start_gpu_monitor():
    samples, stop = [], threading.Event()
    def _fn():
        import subprocess as sp
        while not stop.is_set():
            r = sp.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                        '--format=csv,noheader,nounits'], capture_output=True, text=True)
            if r.returncode == 0:
                try:
                    u, m = r.stdout.strip().split(', ')
                    samples.append((int(u), float(m) / 1024))
                except Exception:
                    pass
            time.sleep(0.5)
    threading.Thread(target=_fn, daemon=True).start()
    return samples, stop

def _launch_count(store, n_active=PROF_ACTIVE):
    if not store.get('avgs'): return None
    e = next((e for e in store['avgs'] if e.key == 'cudaLaunchKernel'), None)
    return e.count // n_active if e else None

def _compiled_region_count(store):
    if not store.get('avgs'): return 0
    return len([e for e in store['avgs'] if 'Torch-Compiled Region' in e.key])

def _detect_attention_kernel(store, label):
    if not (DEVICE == 'cuda' and store.get('avgs')):
        print(f'{label}: no CUDA data'); return f'{label}: no CUDA data'
    flash = next((e for e in store['avgs'] if 'flash_attention' in e.key), None)
    eff   = next((e for e in store['avgs'] if 'efficient_attention' in e.key), None)
    if flash and eff: msg = f'{label}: BOTH flash ({flash.count}) + efficient ({eff.count})'
    elif flash:       msg = f'{label}: FlashAttention ACTIVE ✓ ({flash.count} calls)'
    elif eff:         msg = f'{label}: memory-efficient only ({eff.count} calls)'
    else:             msg = f'{label}: attention kernels not visible (expected for AOTI — runs in C++)'
    print(msg); return msg

def _save(obj, name):
    path = os.path.join(RESULTS_DIR, name)
    if hasattr(obj, 'savefig'): obj.savefig(path, dpi=150, bbox_inches='tight')
    else:                       obj.to_csv(path, index=False)
    print(f'Saved: {path}')
    return path

_tgt = lambda ms: 'MEETS ✓' if ms <= 10 else f'FAILS ({ms:.1f}ms)'

print(f'\nDevice : {DEVICE} | {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: {BF16_SUPPORTED}')
print(f'Batch sizes: {BATCH_SIZES} | N_PEAKS={N_PEAKS} (static, required by AOTI)')
print(f'Artifacts → {AOTI_DIR}')

depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
  flash_compatible param available : ✓

── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed  → _nar_embed        : ✓
  Spec2Pep._forward_step patched            : ✓
  Spec2Pep.forward patched                  : ✓
  mode                                      : flash_compatible=True
──────────────────────────────────────────────────────────────

Device : cuda | NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8 | BF16: True
Batch sizes: [1, 8, 32, 128, 512] | N_PEAKS=150 (static, required by AOTI)
Artifacts → /teamspace/studios/this_studio/aoti_profiling/artifacts


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Data + model + torch.export wrapper modules
# FIX: make_example_inputs() previously used precursors = zeros(bs, 3),
# meaning charge=0. Casanovo's global_token_hook uses charge as an index
# into a charge-embedding table (valid charges start at 1), so charge=0
# was an out-of-bounds embedding lookup — the actual cause of the CUDA
# indexSelectSmallIndex assertion (reported asynchronously, which is why
# the traceback pointed at positional_encoder rather than the real site).
# Precursor tensor layout confirmed from earlier session: [mass, charge, mz].
# mzs/ints zero-tensors are unaffected — they pass through a continuous
# sinusoidal/linear peak encoder, not an embedding lookup, so zero is a
# valid (padding-equivalent) value there and needs no change.
# ═══════════════════════════════════════════════════════════════════════
from casanovo.denovo import ModelRunner
from casanovo.config import Config
from casanovo.casanovo import _get_model_weights
import appdirs

try:
    from casanovo.denovo.dataloaders import DeNovoDataModule
except ModuleNotFoundError:
    from casanovo.data.datasets import DeNovoDataModule

MGF_FILE   = 'multi-enzyme-simple.test.mgf'
SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = '.lance_cache'

for _f in (MGF_FILE, SUBSET_MGF):
    if not os.path.exists(_f):
        raise FileNotFoundError(f'{_f} not found in {WORK_DIR}')
print(f'{MGF_FILE}   ({os.path.getsize(MGF_FILE)/1e6:.1f} MB)')
print(f'{SUBSET_MGF} ({os.path.getsize(SUBSET_MGF)/1e6:.1f} MB)')

# ── Model ────────────────────────────────────────────────────────────
config     = Config(None)
cache_dir  = Path(appdirs.user_cache_dir('casanovo', False, opinion=False))
model_path = _get_model_weights(cache_dir)
print(f'\nCheckpoint: {model_path}')

runner = ModelRunner(config, model_path)
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model = runner.model.eval().to(DEVICE)
MODEL_MAX_CHARGE = getattr(model, 'max_charge', config.max_charge)
MAX_PEP_LEN      = model.max_peptide_len

print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params | '
      f'max_peptide_len={MAX_PEP_LEN} | max_charge={MODEL_MAX_CHARGE}')
assert PeptideDecoder.embed is _nar_embed, 'NAR patch lost after model load!'
print('NAR patch survives model load ✓')

# ── Export wrapper: fused encoder + decoder, positional args only ───────
class FullNARWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.encoder = m.encoder
        self.decoder = m.decoder

    def forward(self, mzs, ints, precursors, tokens):
        memory, mem_mask = self.encoder(mzs, ints)
        return self.decoder(tokens=tokens, memory=memory,
                            memory_key_padding_mask=mem_mask,
                            precursors=precursors)

class FullNARWrapperBF16(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.encoder = m.encoder
        self.decoder = m.decoder

    def forward(self, mzs, ints, precursors, tokens):
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            memory, mem_mask = self.encoder(mzs, ints)
            scores = self.decoder(tokens=tokens, memory=memory,
                                  memory_key_padding_mask=mem_mask,
                                  precursors=precursors)
        return scores.float()

def make_example_inputs(bs):
    """Exact (size, dtype, stride) contract the AOTI artifact will enforce.
    FIX: precursors now uses a realistic charge (2.0) and correctly derived
    neutral mass — charge=0 (the old all-zeros version) was an invalid
    embedding index and crashed the CUDA kernel."""
    mzs   = torch.zeros(bs, N_PEAKS, dtype=torch.float32, device=DEVICE)
    ints  = torch.zeros(bs, N_PEAKS, dtype=torch.float32, device=DEVICE)

    charge = 2.0
    pmz    = 600.0
    neutral_mass = (pmz - 1.007276) * charge
    precs = torch.tensor([[neutral_mass, charge, pmz]] * bs,
                         dtype=torch.float32, device=DEVICE).contiguous()

    toks = torch.zeros(bs, MAX_PEP_LEN, dtype=torch.long, device=DEVICE)
    return (mzs.contiguous(), ints.contiguous(), precs, toks.contiguous())

def make_datamodule(bs):
    dm = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                          eval_batch_size=bs, tokenizer=runner.tokenizer,
                          max_charge=MODEL_MAX_CHARGE, n_workers=0)
    dm.setup(stage='test', annotated=False)
    return dm

# ── Smoke-test both wrappers eagerly before any export ─────────────────
_w_fp32 = FullNARWrapper(model).eval()
_ex = make_example_inputs(1)
with torch.no_grad():
    _out = _w_fp32(*_ex)
print(f'\nFullNARWrapper eager output: {tuple(_out.shape)}  dtype={_out.dtype} ✓')

if BF16_SUPPORTED:
    _w_bf16 = FullNARWrapperBF16(model).eval()
    with torch.no_grad():
        _out_bf = _w_bf16(*_ex)
    print(f'FullNARWrapperBF16 eager output: {tuple(_out_bf.shape)}  dtype={_out_bf.dtype} ✓')

_dm_check = make_datamodule(1)
_b = next(iter(_dm_check.predict_dataloader()))
_mz0, _it0, _pr0, _ = model._process_batch(_b)
print(f'First real batch: mzs={tuple(_mz0.shape)}  precs={tuple(_pr0.shape)} ✓')
print(f'Subset ready: {N_SUBSET} spectra (timing target: {N_TIMING_SPECTRA})')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: [Ammonia-loss]-, C[Carbamidomethyl], [Acetyl]-, [Carbamyl]-, M[Oxidation], N[Deamidated], Q[Deamidated], [+25.980265]-


multi-enzyme-simple.test.mgf   (300.9 MB)
subset_profile.mgf (16.9 MB)

Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt
Model: 47.9M params | max_peptide_len=100 | max_charge=4
NAR patch survives model load ✓

FullNARWrapper eager output: (1, 101, 29)  dtype=torch.float32 ✓
FullNARWrapperBF16 eager output: (1, 101, 29)  dtype=torch.float32 ✓


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

First real batch: mzs=(1, 42)  precs=(1, 3) ✓
Subset ready: 6000 spectra (timing target: 5000)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — Baselines to beat
#   A) FP32 eager
#   B) torch.compile(reduce-overhead) + BF16   ← current best (13.07ms @ bs=1)
# Both use the same fixed-peak padding as AOTI, so the comparison is fair.
# ═══════════════════════════════════════════════════════════════════════
def run_timing(label, fwd_fn, use_bf16=False, use_mark_step=False, warmup_hook=None):
    """fwd_fn(mzs, ints, precs, toks) -> scores.  Returns per-bs timing dict."""
    gpu_s, gpu_stop = _start_gpu_monitor()
    out = {}
    for bs in BATCH_SIZES:
        print(f'\n══ {label}  batch_size={bs:4d} ══')
        dm = make_datamodule(bs)

        t_compile = 0.0
        if warmup_hook is not None:
            t0c = time.perf_counter()
            warmup_hook(bs)
            t_compile = time.perf_counter() - t0c

        # warm-up
        ctx = _bf16_ctx if use_bf16 else (lambda: torch.autocast('cuda', enabled=False))
        with torch.no_grad():
            for w, wb in enumerate(iter(dm.predict_dataloader())):
                if w >= N_WARMUP_BATCHES: break
                wm, wi, wp, _ = model._process_batch(wb)
                wm, wi, wp = wm.to(DEVICE), wi.to(DEVICE), wp.to(DEVICE)
                wm, wi, _ = pad_or_select_to_fixed_peaks(wm, wi)
                wt = torch.zeros((wm.shape[0], MAX_PEP_LEN), dtype=torch.long, device=DEVICE)
                if use_mark_step: _mark_step()
                with ctx():
                    fwd_fn(wm, wi, wp.contiguous(), wt)
        _sync()

        t = {k: [] for k in ['fetch','h2d','fwd','write','total','tp']}
        loader_iter = iter(dm.predict_dataloader()); n_spec = 0
        pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
        while n_spec < N_TIMING_SPECTRA:
            _sync(); t0 = time.perf_counter()
            try: batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(dm.predict_dataloader()); batch = next(loader_iter)
            t_fetch = (time.perf_counter() - t0) * 1000

            _sync(); t0 = time.perf_counter()
            mzs, ints, precs, _ = model._process_batch(batch)
            mzs, ints, precs = mzs.to(DEVICE), ints.to(DEVICE), precs.to(DEVICE).contiguous()
            mzs, ints, _ = pad_or_select_to_fixed_peaks(mzs, ints)
            _sync(); t_h2d = (time.perf_counter() - t0) * 1000
            ab = mzs.shape[0]
            toks = torch.zeros((ab, MAX_PEP_LEN), dtype=torch.long, device=DEVICE)

            with torch.no_grad():
                if use_mark_step: _mark_step()
                _sync(); t0 = time.perf_counter()
                with ctx():
                    scores = fwd_fn(mzs, ints, precs, toks)
                _sync(); t_fwd = (time.perf_counter() - t0) * 1000

            t0 = time.perf_counter()
            pred = scores.argmax(dim=-1).cpu(); _ = [{'tokens': x.tolist()} for x in pred]
            t_write = (time.perf_counter() - t0) * 1000

            tt = t_fetch + t_h2d + t_fwd + t_write
            for k, v in zip(['fetch','h2d','fwd','write','total','tp'],
                            [t_fetch/ab, t_h2d/ab, t_fwd/ab, t_write/ab, tt/ab, ab/(tt/1000)]):
                t[k].append(v)
            n_spec += ab; pbar.update(ab)
            if n_spec >= N_TIMING_SPECTRA: break
        pbar.close()

        p = lambda a, q: float(np.percentile(a, q))
        out[bs] = {'n_spec': n_spec, 'compile_s': t_compile,
                   'fetch': np.mean(t['fetch']), 'h2d': np.mean(t['h2d']),
                   'fwd': np.mean(t['fwd']), 'write': np.mean(t['write']),
                   'total': np.mean(t['total']), 'p50': p(t['total'],50),
                   'p95': p(t['total'],95), 'tp': np.mean(t['tp']), 'raw': t}
        s = out[bs]
        print(f'  total={s["total"]:.2f}ms  fwd={s["fwd"]:.2f}ms  tp={s["tp"]:.1f}spec/s'
              + (f'  compile={t_compile:.1f}s' if t_compile > 0.05 else ''))
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    gpu_stop.set(); time.sleep(1.0)
    util = np.mean([x[0] for x in gpu_s]) if gpu_s else 0
    vram = np.max([x[1] for x in gpu_s]) if gpu_s else 0
    return out, util, vram

# ── A) FP32 eager baseline ─────────────────────────────────────────────
_w_fp32 = FullNARWrapper(model).eval()
timing_fp32, util_fp32, vram_fp32 = run_timing(
    'FP32 eager', lambda a,b,c,d: _w_fp32(a,b,c,d), use_bf16=False)

# ── B) torch.compile(reduce-overhead) + BF16 ──────────────────────────
torch._dynamo.reset()
torch._dynamo.config.cache_size_limit = 32
_compiled_enc = torch.compile(model.encoder, mode='reduce-overhead')
_compiled_dec = torch.compile(model.decoder, mode='reduce-overhead')
print('\ncompiled_encoder / compiled_decoder created (mode=reduce-overhead)')

def _compiled_fwd(mzs, ints, precs, toks):
    mem, mask = _compiled_enc(mzs, ints)
    return _compiled_dec(tokens=toks, memory=mem,
                         memory_key_padding_mask=mask, precursors=precs)

timing_comp, util_comp, vram_comp = run_timing(
    'Compiled+BF16', _compiled_fwd, use_bf16=True, use_mark_step=True)

print('\n── Baselines (bs=1) ─────────────────────────────────────────')
print(f'  FP32 eager     : {timing_fp32[1]["total"]:.2f} ms  ({timing_fp32[1]["tp"]:.1f} spec/s)')
print(f'  Compiled+BF16  : {timing_comp[1]["total"]:.2f} ms  ({timing_comp[1]["tp"]:.1f} spec/s)  '
      f'[{timing_fp32[1]["total"]/timing_comp[1]["total"]:.2f}× vs FP32]')
print(f'  10ms target    : {_tgt(timing_comp[1]["total"])}')


══ FP32 eager  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:45<00:00, 47.18spec/s]


  total=20.81ms  fwd=18.88ms  tp=48.6spec/s

══ FP32 eager  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:05, 1017.39spec/s]                        


  total=0.97ms  fwd=0.74ms  tp=1033.5spec/s

══ Compiled+BF16  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:04, 1125.48spec/s]                        


  total=0.89ms  fwd=0.70ms  tp=1129.8spec/s

── Baselines (bs=1) ─────────────────────────────────────────
  FP32 eager     : 20.81 ms  (48.6 spec/s)
  Compiled+BF16  : 13.04 ms  (77.8 spec/s)  [1.60× vs FP32]
  10ms target    : FAILS (13.0ms)


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — torch.export → AOTInductor compile → .pt2 package
#
# Official API (PyTorch docs, torch.compiler_aot_inductor):
#   ep   = torch.export.export(mod, args)
#   path = torch._inductor.aoti_compile_and_package(ep, package_path=...)
#   fn   = torch._inductor.aoti_load_package(path)
#
# STATIC shapes, one artifact per batch size. Rationale: torch.export
# specializes dims of size 0/1, so a "dynamic" batch dim exported from a
# bs=1 example would specialize to 1 anyway. Static export also avoids all
# dynamic-shape guard failures. Compile cost is a one-time offline cost —
# precisely AOTInductor's intended tradeoff — and is excluded from timings.
#
# strict=True is tried first (Dynamo tracing, strongest soundness guarantee).
# torch.export FORBIDS graph breaks outright (unlike torch.compile, which
# merely fragments), so if depthcharge still has untraceable control flow
# this will raise — we then retry with strict=False (non-strict tracing).
# ═══════════════════════════════════════════════════════════════════════
import torch._inductor

def export_and_compile(wrapper, bs, tag):
    """Returns (loaded_callable, compile_seconds, mode_str) or (None, 0, reason)."""
    ex = make_example_inputs(bs)
    pkg = os.path.join(AOTI_DIR, f'nar_{tag}_bs{bs}.pt2')
    t0 = time.perf_counter()

    ep, mode = None, None
    for strict_flag in (True, False):
        try:
            with torch.no_grad():
                ep = torch.export.export(wrapper, ex, strict=strict_flag)
            mode = f'strict={strict_flag}'
            break
        except Exception as e:
            if strict_flag is True:
                print(f'    strict=True export failed ({type(e).__name__}) → retrying strict=False')
                print(f'      {str(e)[:160]}')
            else:
                return None, 0.0, f'export failed: {type(e).__name__}: {str(e)[:160]}'
    if ep is None:
        return None, 0.0, 'export produced no ExportedProgram'

    try:
        path = torch._inductor.aoti_compile_and_package(ep, package_path=pkg)
        loaded = torch._inductor.aoti_load_package(path)
    except Exception as e:
        return None, 0.0, f'AOTI compile/load failed: {type(e).__name__}: {str(e)[:160]}'

    dt = time.perf_counter() - t0

    # Correctness gate: AOTI output must match eager output on a real batch.
    with torch.no_grad():
        ref = wrapper(*ex)
        got = loaded(*ex)
    max_diff = (ref.float() - got.float()).abs().max().item()
    ok = torch.allclose(ref.float(), got.float(), rtol=1e-3, atol=1e-3)
    print(f'    {tag} bs={bs}: {mode} | compile {dt:5.1f}s | '
          f'max_diff={max_diff:.2e} | {"✓ matches eager" if ok else "✗ MISMATCH"}')
    if not ok:
        return None, dt, f'numerical mismatch (max_diff={max_diff:.2e})'
    return loaded, dt, mode

# ── FP32 AOTI ─────────────────────────────────────────────────────────
print('── Exporting + compiling AOTI (FP32) ────────────────────────')
_w_fp32_exp = FullNARWrapper(model).eval()
aoti_fp32, aoti_fp32_compile = {}, {}
for bs in BATCH_SIZES:
    fn, dt, info = export_and_compile(_w_fp32_exp, bs, 'fp32')
    if fn is not None:
        aoti_fp32[bs] = fn; aoti_fp32_compile[bs] = dt
    else:
        print(f'    fp32 bs={bs}: SKIPPED — {info}')
AOTI_FP32_OK = len(aoti_fp32) == len(BATCH_SIZES)

# ── BF16 AOTI ─────────────────────────────────────────────────────────
aoti_bf16, aoti_bf16_compile = {}, {}
if BF16_SUPPORTED:
    print('\n── Exporting + compiling AOTI (BF16 autocast in graph) ──────')
    _w_bf16_exp = FullNARWrapperBF16(model).eval()
    for bs in BATCH_SIZES:
        fn, dt, info = export_and_compile(_w_bf16_exp, bs, 'bf16')
        if fn is not None:
            aoti_bf16[bs] = fn; aoti_bf16_compile[bs] = dt
        else:
            print(f'    bf16 bs={bs}: SKIPPED — {info}')
AOTI_BF16_OK = len(aoti_bf16) == len(BATCH_SIZES)

print('\n── AOTI compile summary ─────────────────────────────────────')
print(f'  FP32 artifacts : {len(aoti_fp32)}/{len(BATCH_SIZES)}  {"✓" if AOTI_FP32_OK else "(partial)"}')
print(f'  BF16 artifacts : {len(aoti_bf16)}/{len(BATCH_SIZES)}  {"✓" if AOTI_BF16_OK else "(partial)"}')
if aoti_fp32_compile:
    print(f'  FP32 compile   : {sum(aoti_fp32_compile.values()):.1f}s total (one-time, excluded from timings)')
if aoti_bf16_compile:
    print(f'  BF16 compile   : {sum(aoti_bf16_compile.values()):.1f}s total (one-time, excluded from timings)')
if not (AOTI_FP32_OK or AOTI_BF16_OK):
    print('\n  ⚠ No AOTI artifact compiled — downstream cells will skip AOTI and')
    print('    report only the baselines. The failure reasons above are the result.')

── Exporting + compiling AOTI (FP32) ────────────────────────


E0709 13:46:28.878000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.
E0709 13:46:28.879000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.


    strict=True export failed (TorchRuntimeError) → retrying strict=False
      Dynamo failed to run FX node with fake tensors: call_module L__self___encoder_transformer_encoder(*(FakeTensor(..., device='cuda:0', size=(1, 151, 512)),), **{'
    fp32 bs=1: strict=False | compile  46.6s | max_diff=2.08e-02 | ✗ MISMATCH
    fp32 bs=1: SKIPPED — numerical mismatch (max_diff=2.08e-02)


E0709 13:47:14.913000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.
E0709 13:47:14.915000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.


    strict=True export failed (TorchRuntimeError) → retrying strict=False
      Dynamo failed to run FX node with fake tensors: call_module L__self___encoder_transformer_encoder(*(FakeTensor(..., device='cuda:0', size=(8, 151, 512)),), **{'
    fp32 bs=8: strict=False | compile  42.6s | max_diff=2.08e-02 | ✗ MISMATCH
    fp32 bs=8: SKIPPED — numerical mismatch (max_diff=2.08e-02)


...

    strict=True export failed (TorchRuntimeError) → retrying strict=False
      Dynamo failed to run FX node with fake tensors: call_module L__self___encoder_transformer_encoder(*(FakeTensor(..., device='cuda:0', size=(32, 151, 512)),), **{
    fp32 bs=32: strict=False | compile  47.5s | max_diff=2.08e-02 | ✗ MISMATCH
    fp32 bs=32: SKIPPED — numerical mismatch (max_diff=2.08e-02)


E0709 13:48:45.232000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.
E0709 13:48:45.233000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.


    strict=True export failed (TorchRuntimeError) → retrying strict=False
      Dynamo failed to run FX node with fake tensors: call_module L__self___encoder_transformer_encoder(*(FakeTensor(..., device='cuda:0', size=(128, 151, 512)),), **
    fp32 bs=128: strict=False | compile  44.8s | max_diff=2.08e-02 | ✗ MISMATCH
    fp32 bs=128: SKIPPED — numerical mismatch (max_diff=2.08e-02)


E0709 13:49:30.426000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.
E0709 13:49:30.428000 77610 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/export/_trace.py:1079] always_classified is unsupported.


    strict=True export failed (TorchRuntimeError) → retrying strict=False
      Dynamo failed to run FX node with fake tensors: call_module L__self___encoder_transformer_encoder(*(FakeTensor(..., device='cuda:0', size=(512, 151, 512)),), **
    fp32 bs=512: strict=False | compile  44.2s | max_diff=2.08e-02 | ✗ MISMATCH
    fp32 bs=512: SKIPPED — numerical mismatch (max_diff=2.08e-02)

── Exporting + compiling AOTI (BF16 autocast in graph) ──────
    bf16 bs=1: strict=True | compile  46.4s | max_diff=1.18e-01 | ✗ MISMATCH
    bf16 bs=1: SKIPPED — numerical mismatch (max_diff=1.18e-01)
    bf16 bs=8: strict=True | compile  46.2s | max_diff=1.33e-01 | ✗ MISMATCH
    bf16 bs=8: SKIPPED — numerical mismatch (max_diff=1.33e-01)
    bf16 bs=32: strict=True | compile  49.4s | max_diff=1.33e-01 | ✗ MISMATCH
    bf16 bs=32: SKIPPED — numerical mismatch (max_diff=1.33e-01)
    bf16 bs=128: strict=True | compile  49.8s | max_diff=1.10e-01 | ✗ MISMATCH
    bf16 bs=128: SKIPPED — numerical mismatch 

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — AOTI timing, identical methodology to the baselines
# No _mark_step(): AOTI manages its own memory, and CUDA Graph Trees'
# cross-invocation buffer bookkeeping does not apply to the AOTI runtime.
# ═══════════════════════════════════════════════════════════════════════
def run_timing_aoti(label, artifacts, compile_times):
    gpu_s, gpu_stop = _start_gpu_monitor()
    out = {}
    for bs in BATCH_SIZES:
        if bs not in artifacts:
            print(f'\n══ {label}  bs={bs}: no artifact — skipped ══'); continue
        print(f'\n══ {label}  batch_size={bs:4d} ══')
        fn = artifacts[bs]
        dm = make_datamodule(bs)

        with torch.no_grad():
            for w, wb in enumerate(iter(dm.predict_dataloader())):
                if w >= N_WARMUP_BATCHES: break
                wm, wi, wp, _ = model._process_batch(wb)
                wm, wi, wp = wm.to(DEVICE), wi.to(DEVICE), wp.to(DEVICE).contiguous()
                wm, wi, _ = pad_or_select_to_fixed_peaks(wm, wi)
                if wm.shape[0] != bs: continue          # AOTI requires exact bs
                wt = torch.zeros((bs, MAX_PEP_LEN), dtype=torch.long, device=DEVICE)
                fn(wm, wi, wp, wt)
        _sync()

        t = {k: [] for k in ['fetch','h2d','fwd','write','total','tp']}
        loader_iter = iter(dm.predict_dataloader()); n_spec = 0
        pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
        while n_spec < N_TIMING_SPECTRA:
            _sync(); t0 = time.perf_counter()
            try: batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(dm.predict_dataloader()); batch = next(loader_iter)
            t_fetch = (time.perf_counter() - t0) * 1000

            _sync(); t0 = time.perf_counter()
            mzs, ints, precs, _ = model._process_batch(batch)
            mzs, ints, precs = mzs.to(DEVICE), ints.to(DEVICE), precs.to(DEVICE).contiguous()
            mzs, ints, _ = pad_or_select_to_fixed_peaks(mzs, ints)
            _sync(); t_h2d = (time.perf_counter() - t0) * 1000
            ab = mzs.shape[0]
            if ab != bs:            # final partial batch: artifact shape won't match
                continue
            toks = torch.zeros((ab, MAX_PEP_LEN), dtype=torch.long, device=DEVICE)

            with torch.no_grad():
                _sync(); t0 = time.perf_counter()
                scores = fn(mzs, ints, precs, toks)
                _sync(); t_fwd = (time.perf_counter() - t0) * 1000

            t0 = time.perf_counter()
            pred = scores.argmax(dim=-1).cpu(); _ = [{'tokens': x.tolist()} for x in pred]
            t_write = (time.perf_counter() - t0) * 1000

            tt = t_fetch + t_h2d + t_fwd + t_write
            for k, v in zip(['fetch','h2d','fwd','write','total','tp'],
                            [t_fetch/ab, t_h2d/ab, t_fwd/ab, t_write/ab, tt/ab, ab/(tt/1000)]):
                t[k].append(v)
            n_spec += ab; pbar.update(ab)
            if n_spec >= N_TIMING_SPECTRA: break
        pbar.close()

        p = lambda a, q: float(np.percentile(a, q))
        out[bs] = {'n_spec': n_spec, 'compile_s': compile_times.get(bs, 0.0),
                   'fetch': np.mean(t['fetch']), 'h2d': np.mean(t['h2d']),
                   'fwd': np.mean(t['fwd']), 'write': np.mean(t['write']),
                   'total': np.mean(t['total']), 'p50': p(t['total'],50),
                   'p95': p(t['total'],95), 'tp': np.mean(t['tp']), 'raw': t}
        s = out[bs]
        base = timing_fp32[bs]['total']
        print(f'  total={s["total"]:.2f}ms  fwd={s["fwd"]:.2f}ms  tp={s["tp"]:.1f}spec/s  '
              f'vs FP32: {base/max(s["total"],1e-3):.2f}×')
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    gpu_stop.set(); time.sleep(1.0)
    util = np.mean([x[0] for x in gpu_s]) if gpu_s else 0
    vram = np.max([x[1] for x in gpu_s]) if gpu_s else 0
    return out, util, vram

timing_aoti_fp32, util_aoti_fp32, vram_aoti_fp32 = ({}, 0, 0)
timing_aoti_bf16, util_aoti_bf16, vram_aoti_bf16 = ({}, 0, 0)

if aoti_fp32:
    timing_aoti_fp32, util_aoti_fp32, vram_aoti_fp32 = run_timing_aoti(
        'AOTI FP32', aoti_fp32, aoti_fp32_compile)
if aoti_bf16:
    timing_aoti_bf16, util_aoti_bf16, vram_aoti_bf16 = run_timing_aoti(
        'AOTI BF16', aoti_bf16, aoti_bf16_compile)

# ── bs=1 head-to-head ─────────────────────────────────────────────────
print('\n── bs=1 head-to-head ────────────────────────────────────────')
rows = [('FP32 eager', timing_fp32), ('Compiled+BF16 (current best)', timing_comp)]
if timing_aoti_fp32: rows.append(('AOTI FP32', timing_aoti_fp32))
if timing_aoti_bf16: rows.append(('AOTI BF16', timing_aoti_bf16))
base1 = timing_fp32[1]['total']
print(f'{"Variant":<32} {"ms":>8} {"spec/s":>9} {"vs FP32":>9} {"10ms":>8}')
print('-' * 70)
for name, tm in rows:
    if 1 not in tm: continue
    v = tm[1]
    print(f'{name:<32} {v["total"]:>8.2f} {v["tp"]:>9.1f} '
          f'{base1/max(v["total"],1e-3):>8.2f}× {_tgt(v["total"]):>8}')

_best = min((tm[1]['total'], n) for n, tm in rows if 1 in tm)
print(f'\nBest at bs=1: {_best[1]} — {_best[0]:.2f} ms')


── bs=1 head-to-head ────────────────────────────────────────
Variant                                ms    spec/s   vs FP32     10ms
----------------------------------------------------------------------
FP32 eager                          20.81      48.6     1.00× FAILS (20.8ms)
Compiled+BF16 (current best)        13.04      77.8     1.60× FAILS (13.0ms)

Best at bs=1: Compiled+BF16 (current best) — 13.04 ms


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — torch.profiler (bs=1, 50 profiled spectra)
#
# The decisive metric is cudaLaunchKernel count per spectrum. AOTI executes
# in C++, so individual aten:: ops are NOT visible in the trace — that
# absence is itself the evidence that Python-side dispatch was eliminated.
# ═══════════════════════════════════════════════════════════════════════
ACTS   = ([ProfilerActivity.CPU, ProfilerActivity.CUDA] if DEVICE == 'cuda'
          else [ProfilerActivity.CPU])
N_PROF = PROF_WARMUP + PROF_ACTIVE

print(f'Pre-fetching {N_PROF} bs=1 batches (fixed to N_PEAKS={N_PEAKS})…')
_dm_prof = make_datamodule(1)
prof_batches = []
for b in _dm_prof.predict_dataloader():
    mz, it, pr, _ = model._process_batch(b)
    mz, it = mz.to(DEVICE), it.to(DEVICE)
    mz, it, _ = pad_or_select_to_fixed_peaks(mz, it)
    tk = torch.zeros((1, MAX_PEP_LEN), dtype=torch.long, device=DEVICE)
    prof_batches.append((mz, it, pr.to(DEVICE).contiguous(), tk))
    if len(prof_batches) >= N_PROF: break
while len(prof_batches) < N_PROF:
    prof_batches.extend(prof_batches[:N_PROF - len(prof_batches)])
print(f'Using {len(prof_batches)} batches')

def run_profiler(label, fwd_fn, trace_name, use_bf16=False, use_mark_step=False, n_warm=10):
    ctx = _bf16_ctx if use_bf16 else (lambda: torch.autocast('cuda', enabled=False))
    with torch.no_grad():
        for mz, it, pr, tk in prof_batches[:n_warm]:
            if use_mark_step: _mark_step()
            with ctx(): fwd_fn(mz, it, pr, tk)
    _sync()

    store = {}
    trace_path = os.path.join(RESULTS_DIR, trace_name)
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        store['tbl']  = p.key_averages().table(sort_by='cpu_time_total', row_limit=12)
        store['avgs'] = p.key_averages()

    with profile(activities=ACTS, record_shapes=True,
                 schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
                 on_trace_ready=_on_ready) as p:
        with torch.no_grad():
            for mz, it, pr, tk in prof_batches:
                if use_mark_step: _mark_step()
                with record_function(label):
                    with ctx(): fwd_fn(mz, it, pr, tk)
                _sync(); p.step()

    print(store.get('tbl', '(no data)'))
    txt = os.path.join(RESULTS_DIR, trace_name.replace('trace_', 'profiler_').replace('.json', '.txt'))
    with open(txt, 'w') as f:
        f.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        f.write('=' * 64 + '\n' + str(store.get('tbl', 'no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE == 'cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()
    return store

prof = {}

print('\n── A) FP32 eager ───────────────────────────────────────────')
prof['fp32'] = run_profiler('fp32_eager', lambda a,b,c,d: _w_fp32(a,b,c,d),
                            'trace_aoti_cmp_fp32.json')
attn_fp32 = _detect_attention_kernel(prof['fp32'], 'A) FP32 eager')

print('\n── B) Compiled + BF16 ──────────────────────────────────────')
prof['comp'] = run_profiler('compiled_bf16', _compiled_fwd, 'trace_aoti_cmp_compiled.json',
                            use_bf16=True, use_mark_step=True)
attn_comp = _detect_attention_kernel(prof['comp'], 'B) Compiled+BF16')

if 1 in aoti_fp32:
    print('\n── C) AOTI FP32 ────────────────────────────────────────────')
    prof['aoti_fp32'] = run_profiler('aoti_fp32', aoti_fp32[1], 'trace_aoti_fp32.json')
    attn_aoti_fp32 = _detect_attention_kernel(prof['aoti_fp32'], 'C) AOTI FP32')

if 1 in aoti_bf16:
    print('\n── D) AOTI BF16 ────────────────────────────────────────────')
    prof['aoti_bf16'] = run_profiler('aoti_bf16', aoti_bf16[1], 'trace_aoti_bf16.json')
    attn_aoti_bf16 = _detect_attention_kernel(prof['aoti_bf16'], 'D) AOTI BF16')

print('\n── Profiler comparison (bs=1, 50 spectra) ───────────────────')
print(f'{"Variant":<22} {"launches/spec":>15} {"compiled regions":>18}')
print('-' * 58)
_names = {'fp32': 'FP32 eager', 'comp': 'Compiled+BF16',
          'aoti_fp32': 'AOTI FP32', 'aoti_bf16': 'AOTI BF16'}
launches = {}
for k, s in prof.items():
    lc = _launch_count(s); rc = _compiled_region_count(s)
    launches[k] = lc
    rc_str = str(rc) if rc else ('n/a (C++ runtime)' if k.startswith('aoti') else 'n/a (eager)')
    print(f'{_names[k]:<22} {str(lc):>15} {rc_str:>18}')

Pre-fetching 70 bs=1 batches (fixed to N_PEAKS=150)…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches

── A) FP32 eager ───────────────────────────────────────────
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.35%       7.869ms       100.00%        2.266s      45.317ms       0.000us         0.00%     152.093ms       3.042ms            50  
                                             fp32_eager        22.83%     517.335ms        99.57%        2.256s      45.121ms   

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Comparison plots (only variants that actually compiled)
# ═══════════════════════════════════════════════════════════════════════
variants = [('FP32 eager', timing_fp32, '#D85A30'),
            ('Compiled+BF16', timing_comp, '#1D9E75')]
if timing_aoti_fp32: variants.append(('AOTI FP32', timing_aoti_fp32, '#3A7FC1'))
if timing_aoti_bf16: variants.append(('AOTI BF16', timing_aoti_bf16, '#8E44AD'))

xi   = list(range(len(BATCH_SIZES)))
xlbl = [str(b) for b in BATCH_SIZES]

# Fig 1 — throughput + latency vs batch size
fig1, (ax_tp, ax_lat) = plt.subplots(1, 2, figsize=(14, 5))
fig1.suptitle('AOTInductor vs torch.compile vs Eager — NAR Casanovo', fontweight='bold')
for name, tm, col in variants:
    xs  = [i for i, b in enumerate(BATCH_SIZES) if b in tm]
    tps = [tm[BATCH_SIZES[i]]['tp'] for i in xs]
    lat = [tm[BATCH_SIZES[i]]['total'] for i in xs]
    ax_tp.plot(xs, tps, 'o-', color=col, lw=2, ms=7, label=name)
    ax_lat.plot(xs, lat, 'o-', color=col, lw=2, ms=7, label=name)
for ax, yl, ti in [(ax_tp, 'Throughput (spec/s)', 'Throughput vs Batch Size'),
                   (ax_lat, 'ms / spectrum', 'Latency vs Batch Size')]:
    ax.set_xticks(xi); ax.set_xticklabels(xlbl)
    ax.set_xlabel('Batch size'); ax.set_ylabel(yl); ax.set_title(ti)
    ax.legend(fontsize=8, frameon=False); ax.spines[['top','right']].set_visible(False)
ax_lat.axhline(10, color='black', lw=1.5, ls=':', label='10 ms (100 Hz target)')
ax_lat.legend(fontsize=8, frameon=False)
plt.tight_layout(); _save(fig1, 'aoti_throughput_latency.png'); plt.show()

# Fig 2 — bs=1 latency bars + kernel launches
fig2, (ax_b, ax_k) = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('bs=1 — Latency and CPU Dispatch Overhead', fontweight='bold')

names = [n for n, tm, _ in variants if 1 in tm]
lats  = [tm[1]['total'] for n, tm, _ in variants if 1 in tm]
cols  = [c for n, tm, c in variants if 1 in tm]
bars = ax_b.bar(names, lats, color=cols, width=0.55, edgecolor='none')
for b, v in zip(bars, lats):
    ax_b.text(b.get_x()+b.get_width()/2, b.get_height()+max(lats)*0.02,
              f'{v:.2f} ms', ha='center', fontsize=9)
ax_b.axhline(10, color='black', lw=1.5, ls=':', label='10 ms target')
ax_b.set_ylabel('ms / spectrum'); ax_b.set_title('Latency @ bs=1')
ax_b.legend(fontsize=8, frameon=False); ax_b.spines[['top','right']].set_visible(False)
plt.setp(ax_b.get_xticklabels(), rotation=15, ha='right')

klabels = [_names[k] for k in prof if launches.get(k) is not None]
kvals   = [launches[k] for k in prof if launches.get(k) is not None]
kcols   = ['#D85A30', '#1D9E75', '#3A7FC1', '#8E44AD'][:len(kvals)]
bars2 = ax_k.bar(klabels, kvals, color=kcols, width=0.55, edgecolor='none')
for b, v in zip(bars2, kvals):
    ax_k.text(b.get_x()+b.get_width()/2, b.get_height()+max(kvals)*0.02,
              f'{v}', ha='center', fontsize=9)
ax_k.set_ylabel('cudaLaunchKernel / spectrum')
ax_k.set_title('CPU Dispatch Overhead @ bs=1 (lower = better)')
ax_k.spines[['top','right']].set_visible(False)
plt.setp(ax_k.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout(); _save(fig2, 'aoti_bs1_comparison.png'); plt.show()

# Fig 3 — bs=1 latency distributions
fig3, ax3 = plt.subplots(figsize=(9, 5))
for name, tm, col in variants:
    if 1 not in tm: continue
    raw = tm[1]['raw']['total']
    ax3.hist(raw, bins=30, alpha=0.5, color=col, label=f'{name} (mean {np.mean(raw):.1f}ms)')
ax3.axvline(10, color='black', lw=1.5, ls=':', label='10 ms target')
ax3.set_xlabel('ms / spectrum'); ax3.set_title('bs=1 Latency Distribution (5000 real spectra)')
ax3.legend(fontsize=8, frameon=False); ax3.spines[['top','right']].set_visible(False)
plt.tight_layout(); _save(fig3, 'aoti_latency_hist.png'); plt.show()

Saved: /teamspace/studios/this_studio/aoti_profiling/results/aoti_throughput_latency.png
Saved: /teamspace/studios/this_studio/aoti_profiling/results/aoti_bs1_comparison.png
Saved: /teamspace/studios/this_studio/aoti_profiling/results/aoti_latency_hist.png


In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Summary + artifact export
# ═══════════════════════════════════════════════════════════════════════
now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

df_all = pd.DataFrame([{
    'batch_size': bs,
    'fp32_eager_ms':    timing_fp32[bs]['total'],
    'compiled_bf16_ms': timing_comp[bs]['total'],
    'aoti_fp32_ms':     timing_aoti_fp32.get(bs, {}).get('total', np.nan),
    'aoti_bf16_ms':     timing_aoti_bf16.get(bs, {}).get('total', np.nan),
    'fp32_tp':          timing_fp32[bs]['tp'],
    'compiled_tp':      timing_comp[bs]['tp'],
    'aoti_fp32_tp':     timing_aoti_fp32.get(bs, {}).get('tp', np.nan),
    'aoti_bf16_tp':     timing_aoti_bf16.get(bs, {}).get('tp', np.nan),
} for bs in BATCH_SIZES]).round(3)
_save(df_all, 'aoti_full_comparison.csv')

b1 = timing_fp32[1]['total']
def _spd(tm): return f"{b1/tm[1]['total']:.2f}×" if 1 in tm else 'n/a'

summary = f"""CASANOVO NAR — AOTInductor / torch.export EXPERIMENT
Generated : {now}
Hardware  : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
depthcharge: {_dc.__file__}
NAR mode  : {'flash_compatible=True' if _HAS_FLASH_COMPAT else 'all-False tgt_mask'}
Dataset   : {SUBSET_MGF} | {N_TIMING_SPECTRA} spectra timed per batch size
Shapes    : N_PEAKS={N_PEAKS} (static), max_peptide_len={MAX_PEP_LEN}

WHAT WAS TESTED
  torch.export.export() captured the FUSED encoder+decoder as a single
  graph (FullNARWrapper), then torch._inductor.aoti_compile_and_package()
  compiled it ahead-of-time into a .pt2 artifact executed by a pure C++
  runtime — no Python dispatch, no Dynamo guard-checking at call time.
  A separate static-shape artifact was built per batch size (torch.export
  specializes size-0/1 dims, so a dynamic batch dim exported from bs=1
  would specialize to 1 regardless).

  Every artifact was gated on numerical correctness against eager output
  (rtol=atol=1e-3) before being timed.

COMPILE STATUS
  AOTI FP32 : {len(aoti_fp32)}/{len(BATCH_SIZES)} artifacts  {'✓' if AOTI_FP32_OK else '(partial)'}
  AOTI BF16 : {len(aoti_bf16)}/{len(BATCH_SIZES)} artifacts  {'✓' if AOTI_BF16_OK else '(partial)'}
  One-time compile cost (excluded from all timings below):
    FP32 {sum(aoti_fp32_compile.values()):.1f}s | BF16 {sum(aoti_bf16_compile.values()):.1f}s

bs=1 RESULTS — the real-time-critical case
  FP32 eager                  : {timing_fp32[1]['total']:6.2f} ms  ({timing_fp32[1]['tp']:6.1f} spec/s)  1.00×
  Compiled+BF16 (prior best)  : {timing_comp[1]['total']:6.2f} ms  ({timing_comp[1]['tp']:6.1f} spec/s)  {_spd(timing_comp)}
"""
if 1 in timing_aoti_fp32:
    summary += f"  AOTI FP32                   : {timing_aoti_fp32[1]['total']:6.2f} ms  ({timing_aoti_fp32[1]['tp']:6.1f} spec/s)  {_spd(timing_aoti_fp32)}\n"
if 1 in timing_aoti_bf16:
    summary += f"  AOTI BF16                   : {timing_aoti_bf16[1]['total']:6.2f} ms  ({timing_aoti_bf16[1]['tp']:6.1f} spec/s)  {_spd(timing_aoti_bf16)}\n"

summary += f"""
  10 ms (100 Hz) target: {_tgt(min(tm[1]['total'] for _, tm, _ in variants if 1 in tm))}

CPU DISPATCH OVERHEAD (bs=1, 50 profiled spectra)
"""
for k in prof:
    if launches.get(k) is not None:
        summary += f"  {_names[k]:<24}: {launches[k]:>5} cudaLaunchKernel / spectrum\n"
summary += """  (AOTI runs in a C++ runtime — individual aten:: ops do not appear in the
   trace. Their absence, together with the launch count, is the evidence
   that Python-side dispatch was removed rather than merely reduced.)

FULL BATCH-SIZE COMPARISON (ms / spectrum)
"""
summary += df_all[['batch_size','fp32_eager_ms','compiled_bf16_ms',
                   'aoti_fp32_ms','aoti_bf16_ms']].to_string(index=False)

summary += f"""

GPU UTILIZATION / PEAK VRAM
  FP32 eager    : {util_fp32:.0f}% | {vram_fp32:.2f} GB
  Compiled+BF16 : {util_comp:.0f}% | {vram_comp:.2f} GB
  AOTI FP32     : {util_aoti_fp32:.0f}% | {vram_aoti_fp32:.2f} GB
  AOTI BF16     : {util_aoti_bf16:.0f}% | {vram_aoti_bf16:.2f} GB

ARTIFACTS
  {RESULTS_DIR}/aoti_throughput_latency.png
  {RESULTS_DIR}/aoti_bs1_comparison.png
  {RESULTS_DIR}/aoti_latency_hist.png
  {RESULTS_DIR}/aoti_full_comparison.csv
  {RESULTS_DIR}/trace_*.json   (open in ui.perfetto.dev)
  {AOTI_DIR}/nar_*.pt2         (deployable AOTI packages)
  {RESULTS_DIR}/aoti_summary.txt
"""
print(summary)
with open(os.path.join(RESULTS_DIR, 'aoti_summary.txt'), 'w') as f:
    f.write(summary)

print('\n── results/ ──')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  {f:<40} {os.path.getsize(os.path.join(RESULTS_DIR, f))/1024:>9.1f} KB')
print('\n── artifacts/ ──')
for f in sorted(os.listdir(AOTI_DIR)):
    print(f'  {f:<40} {os.path.getsize(os.path.join(AOTI_DIR, f))/1024:>9.1f} KB')
print('\nAOTInductor experiment complete.')

Saved: /teamspace/studios/this_studio/aoti_profiling/results/aoti_full_comparison.csv
CASANOVO NAR — AOTInductor / torch.export EXPERIMENT
Generated : 2026-07-09 13:57
Hardware  : NVIDIA L4 | 23.6 GB VRAM | PyTorch 2.7.1+cu128
depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
NAR mode  : flash_compatible=True
Dataset   : subset_profile.mgf | 5000 spectra timed per batch size
Shapes    : N_PEAKS=150 (static), max_peptide_len=100

WHAT WAS TESTED
  torch.export.export() captured the FUSED encoder+decoder as a single
  graph (FullNARWrapper), then torch._inductor.aoti_compile_and_package()
  compiled it ahead-of-time into a .pt2 artifact executed by a pure C++
  runtime — no Python dispatch, no Dynamo guard-checking at call time.
  A separate static-shape artifact was built per batch size (torch.export
  specializes size-0/1 dims, so a dynamic batch dim exported from bs=1
  would specialize to 1 regardless).

  Every artifact was gated on numerical co